### Data Analysis Workflow
#### 1. Reading: from CSV, TXT, JSON, HDF5, SQL, etc.
#### 2. Cleaning: Handle missing values, remove duplicates, detect and address outliers, and standardize data types/formats, etc.
#### 3. Analyzing: Filter data, engineer features, reshape, group and aggregate, join/merge, and apply window operations, etc.
#### 4. Visualization: Create charts and dashboards to communicate insights
#### With pandas, steps 1–3 are well supported; visualization can be done with pandas itself or libraries like Matplotlib and Seaborn.

### Brief Data Overview (Weather.csv)
#### Field Description:
##### - date: date (YYYY-MM-DD)
##### - precipitation: rainfall, most likely in millimeters (mm)
##### - temp_max: daily maximum temperature, most likely in degrees Celsius (°C)
##### - temp_min: daily minimum temperature, most likely in degrees Celsius (°C)
##### - wind: wind speed; unit not specified, possibly meters per second or meters per hour (confirm with the data source)
##### - weather: weather category (e.g., rain, sun, drizzle, snow, fog)
#### Time Range and Completeness:
##### - Coverage dates: 2012-01-01 to 2015-12-31
##### - From 2015-03-03 to 2015-12-30, many records contain missing values (fields left blank); 2015-12-31 has complete data
##### - From 2012-01-01 to 2015-03-02, records are mostly daily and fields are complete

## Pandas File I/O

In [1]:
## Exercise 1
import pandas as pd

#Read data and show data types
df = pd.read_csv("./Weather.csv")
print(df.dtypes)

date              object
precipitation    float64
temp_max         float64
temp_min         float64
wind             float64
weather           object
dtype: object


In [2]:
print(df.head(5)) #Show top 5 rows

         date  precipitation  temp_max  temp_min  wind  weather
0  2012-01-01            0.0      12.8       5.0   4.7  drizzle
1  2012-01-02           10.9      10.6       2.8   4.5     rain
2  2012-01-03            0.8      11.7       7.2   2.3     rain
3  2012-01-04           20.3      12.2       5.6   4.7     rain
4  2012-01-05            1.3       8.9       2.8   6.1     rain


In [3]:
### pd.to_csv() 将数据保存为csv格式文件，数据之间以逗号分隔
### -sep：参数设置使用其他分隔符
### -index：参数设置是否保存行标签
### -header：参数设置是否保存列标签。

#Save the top 6 rows into files
df.iloc[:6].to_csv("df.txt",sep='\t')
df.iloc[:6].to_csv("df2.csv",index=False) #Not save row labels

## Dealing with Missing Values

In [4]:
## Exercise 2
import pandas as pd

#Read data
df = pd.read_csv("./Weather.csv")
print(df.tail(5)) #Show the bottom 5 rows

            date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN       NaN       NaN   NaN     NaN
1457  2015-12-28            NaN       NaN       NaN   NaN     NaN
1458  2015-12-29            NaN       NaN       NaN   NaN     NaN
1459  2015-12-30            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain


In [5]:
### df.isna() 或 df.isnull() 返回布尔矩阵，标记缺失值（NaN 或 None）
print(df.isnull().tail()) #Detect missing values and show the bottom 5 rows
print('='*100)
print('Count NaNs per column:\n', df.isnull().sum()) #Count NaNs per column

       date  precipitation  temp_max  temp_min   wind  weather
1456  False           True      True      True   True     True
1457  False           True      True      True   True     True
1458  False           True      True      True   True     True
1459  False           True      True      True   True     True
1460  False          False     False     False  False    False
Count NaNs per column:
 date               0
precipitation    303
temp_max         303
temp_min         303
wind             303
weather          303
dtype: int64


In [6]:
### df.dropna() 删除包含缺失值的行（默认）
### 设置参数 axis=1 删除包含缺失值的列
### 设置参数 subset=['col1'] 仅删除指定列(例如col1)缺失值所在的行
print('Original shape:', df.shape) #Original shape
print('NaN-rows dropped shape:', df.dropna().shape) #Drop rows with any NaN
print('NaN-cols dropped shape:', df.dropna(axis=1).shape) #Drop columns with any NaN
print('='*100)
print(df.dropna(subset=['weather']).tail()) #Drop rows based on 'weather' column and show the bottom 5 rows

Original shape: (1461, 6)
NaN-rows dropped shape: (1158, 6)
NaN-cols dropped shape: (1461, 1)
            date  precipitation  temp_max  temp_min  wind weather
1153  2015-02-27           18.3      10.0       6.7   4.0    rain
1154  2015-02-28            0.0      12.2       3.3   5.1     sun
1155  2015-03-01            0.0      11.1       1.1   2.2     sun
1156  2015-03-02            0.0      11.1       4.4   4.8     sun
1460  2015-12-31           20.6      12.2       5.0   3.8    rain


In [7]:
#Show and compare different ways to fill DataFrame
print(df.tail()) #Show the bottom 5 rows on original DataFrame
print('='*100)
print(df.fillna({"temp_max": 60, "temp_min": -60}).tail()) #Fill  with constant values by column

            date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN       NaN       NaN   NaN     NaN
1457  2015-12-28            NaN       NaN       NaN   NaN     NaN
1458  2015-12-29            NaN       NaN       NaN   NaN     NaN
1459  2015-12-30            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain
            date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN      60.0     -60.0   NaN     NaN
1457  2015-12-28            NaN      60.0     -60.0   NaN     NaN
1458  2015-12-29            NaN      60.0     -60.0   NaN     NaN
1459  2015-12-30            NaN      60.0     -60.0   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain


In [8]:
### df.ffill()用前一个非缺失值填充（向前填充）
### df.bfill()用后一个非缺失值填充（向后填充）
print('Original DataFrame:\n', df.iloc[1155:1160,:])
print('='*100)
print('Forward filled:\n', df.ffill(inplace=False).iloc[1155:1160,:]) #Forward fill on original DataFrame

Original DataFrame:
             date  precipitation  temp_max  temp_min  wind weather
1155  2015-03-01            0.0      11.1       1.1   2.2     sun
1156  2015-03-02            0.0      11.1       4.4   4.8     sun
1157  2015-03-03            NaN       NaN       NaN   NaN     NaN
1158  2015-03-04            NaN       NaN       NaN   NaN     NaN
1159  2015-03-05            NaN       NaN       NaN   NaN     NaN
Forward filled:
             date  precipitation  temp_max  temp_min  wind weather
1155  2015-03-01            0.0      11.1       1.1   2.2     sun
1156  2015-03-02            0.0      11.1       4.4   4.8     sun
1157  2015-03-03            0.0      11.1       4.4   4.8     sun
1158  2015-03-04            0.0      11.1       4.4   4.8     sun
1159  2015-03-05            0.0      11.1       4.4   4.8     sun


In [ ]:
df = pd.read_csv("./Weather.csv")
print('Original DataFrame:\n', df.tail())
print('='*100)
df.bfill(inplace=True)
print('Backward filled:\n', df.tail()) #Backward fill on original DataFrame

Original DataFrame:
             date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN       NaN       NaN   NaN     NaN
1457  2015-12-28            NaN       NaN       NaN   NaN     NaN
1458  2015-12-29            NaN       NaN       NaN   NaN     NaN
1459  2015-12-30            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain


AttributeError: 'NoneType' object has no attribute 'tail'

In [ ]:
### Fill the NaN values at column 'weather' as 'Unknown'
### Fill the NaN values at columns 'precipitation', 'temp_max', 'temp_min' and 'wind' with given values
df = pd.read_csv("./Weather.csv")
df['weather'] = df['weather'].fillna('Unknown') #With constant value
df['precipitation'] = df['precipitation'].fillna(df['precipitation'].mean()) #Fill 'precipitation' With mean value
df.fillna({"temp_max": df['temp_max'].median(), "temp_min": df['temp_min'].median()}, inplace=True) #Fill 'temp_max' and 'temp_min' with median values
df['wind'] = df['wind'].fillna(df['wind'].mean()) #Fill 'wind' With mean value

print('Filled DataFrame:\n', df.tail())

Filled DataFrame:
             date  precipitation  temp_max  temp_min      wind  weather
1456  2015-12-27       3.052332      14.4       7.8  3.242055  Unknown
1457  2015-12-28       3.052332      14.4       7.8  3.242055  Unknown
1458  2015-12-29       3.052332      14.4       7.8  3.242055  Unknown
1459  2015-12-30       3.052332      14.4       7.8  3.242055  Unknown
1460  2015-12-31      20.600000      12.2       5.0  3.800000     rain


## Dealing with Duplicates

In [ ]:
## Exercise 3
import pandas as pd

#Read data
df = pd.read_csv("./Weather.csv")

In [ ]:
#Add 2 duplicate rows
df = pd.concat([df, df.iloc[[-2, -3]]], ignore_index=True) #Concatenate the original DataFrame and its last two & three rows
print(df.tail())
print('='*100)
### df.duplicated() 返回布尔序列标记重复行（首次出现的行标记为 False）
print('Count duplicate rows:', df.duplicated().sum()) #Count duplicate rows

            date  precipitation  temp_max  temp_min  wind weather
1458  2015-12-29            NaN       NaN       NaN   NaN     NaN
1459  2015-12-30            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain
1461  2015-12-29            NaN       NaN       NaN   NaN     NaN
1462  2015-12-30            NaN       NaN       NaN   NaN     NaN
Count duplicate rows: 2


In [ ]:
### df.drop_duplicates() 保留首次出现的行（默认检查所有列）
### 设置参数subset=['col1'] 仅根据指定列去重
### 设置参数 keep='last' 保留最后一次出现的行
#Remove duplicate rows
print('Keep first:\n', df.drop_duplicates().tail())
print('='*100)
df.drop_duplicates(keep='last',inplace=True) #Delete duplicates on original DataFrame
print('Keep last:\n', df.tail())

Keep first:
             date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN       NaN       NaN   NaN     NaN
1457  2015-12-28            NaN       NaN       NaN   NaN     NaN
1458  2015-12-29            NaN       NaN       NaN   NaN     NaN
1459  2015-12-30            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain
Keep last:
             date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN       NaN       NaN   NaN     NaN
1457  2015-12-28            NaN       NaN       NaN   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain
1461  2015-12-29            NaN       NaN       NaN   NaN     NaN
1462  2015-12-30            NaN       NaN       NaN   NaN     NaN


## Data Conversion

In [ ]:
## Exercise 4
import pandas as pd

#Read data
df = pd.read_csv("./Weather.csv")

In [ ]:
### df['col'].astype() 将指定列转换为某类型（如 int, float, str, category）
#View and convert data types
print(df.dtypes) #View original data types
print('='*100)
df['weather']=df['weather'].astype('category') #Convert 'weather' as category
print(df.dtypes) #Again view data types

date                 str
precipitation    float64
temp_max         float64
temp_min         float64
wind             float64
weather              str
dtype: object
date                  str
precipitation     float64
temp_max          float64
temp_min          float64
wind              float64
weather          category
dtype: object


In [ ]:
#Data checking and filtering
print('Summary before filtering:\n', df['weather'].value_counts()) #Count days of each weather type
print('='*100)
prec_thres = df['precipitation'].mean()+2*df['precipitation'].std() #Precipitation threshold: mean plus two standard deviations
df1 = df[df['precipitation']>prec_thres] #Create a new DataFrame with precipitation values above threshold
print('Summary after filtering:\n', df1['weather'].value_counts()) #Count days of each weather type

Summary before filtering:
 weather
rain       529
sun        490
fog         67
drizzle     46
snow        26
Name: count, dtype: int64
Summary after filtering:
 weather
rain       61
snow        4
drizzle     0
fog         0
sun         0
Name: count, dtype: int64


In [ ]:
print('DataFrame in "rain/snow" days:\n', df[df['weather'].isin(['rain','snow'])]) #Filter data in 'rain' and 'snow' days

DataFrame in "rain/snow" days:
             date  precipitation  temp_max  temp_min  wind weather
1     2012-01-02           10.9      10.6       2.8   4.5    rain
2     2012-01-03            0.8      11.7       7.2   2.3    rain
3     2012-01-04           20.3      12.2       5.6   4.7    rain
4     2012-01-05            1.3       8.9       2.8   6.1    rain
5     2012-01-06            2.5       4.4       2.2   2.2    rain
...          ...            ...       ...       ...   ...     ...
1146  2015-02-20            0.8      11.1       7.2   0.9    rain
1151  2015-02-25            4.1      10.0       6.7   1.0    rain
1152  2015-02-26            9.4      11.7       7.8   1.4    rain
1153  2015-02-27           18.3      10.0       6.7   4.0    rain
1460  2015-12-31           20.6      12.2       5.0   3.8    rain

[555 rows x 6 columns]


In [ ]:
#Convert into frequency
weather_prop = df['weather'].value_counts(normalize=True).to_dict()
print('Encode weather as proportions:\n', weather_prop)
print('='*100)
df['weather_encoded1'] = df['weather'].map(weather_prop)
print('Weather and encoded label1:\n', df[['weather','weather_encoded1']].head())

Encode weather as proportions:
 {'rain': 0.45682210708117443, 'sun': 0.4231433506044905, 'fog': 0.05785837651122625, 'drizzle': 0.039723661485319514, 'snow': 0.022452504317789293}
Weather and encoded label1:
    weather weather_encoded1
0  drizzle         0.039724
1     rain         0.456822
2     rain         0.456822
3     rain         0.456822
4     rain         0.456822


In [ ]:
#Convert into labels
weather_label = dict(zip(df['weather'].unique(),[1,2,3,4,5,6]))
print('Encode weather as integers:\n', weather_label)
print('='*100)
df['weather_encoded2'] = df['weather'].map(weather_label)
print('Weather and encoded label2:\n', df[['weather','weather_encoded2']].head())

Encode weather as integers:
 {'drizzle': 1, 'rain': 2, 'sun': 3, 'snow': 4, 'fog': 5, nan: 6}
Weather and encoded label2:
    weather  weather_encoded2
0  drizzle                 1
1     rain                 2
2     rain                 2
3     rain                 2
4     rain                 2


In [ ]:
#Binarize by threshold
prec_thres = 10
df['prec_bin'] = df['precipitation'].apply(lambda x: 1 if x>prec_thres else 0)
print('Precipitation and binarized label:\n', df[['precipitation','prec_bin']].head())

Precipitation and binarized label:
    precipitation  prec_bin
0            0.0         0
1           10.9         1
2            0.8         0
3           20.3         1
4            1.3         0


In [ ]:
#Standardize data
prec_mean = df['precipitation'].mean() #average
prec_std = df['precipitation'].std() #standard deviation
df['prec_scaled'] = (df['precipitation']-prec_mean)/prec_std
print('Original and z-scaled precipitation:\n', df[['precipitation','prec_scaled']].head())

Original and z-scaled precipitation:
    precipitation  prec_scaled
0            0.0    -0.479935
1           10.9     1.233932
2            0.8    -0.354147
3           20.3     2.711946
4            1.3    -0.275529


## Data Binning and Discretization

In [ ]:
## Exercise 5
import pandas as pd

#Read data
df = pd.read_csv("./Weather.csv")

In [ ]:
#Use cut()
#### pd.cut()方法参数
#### -x：要分箱的数组或Series，通常为数值型
#### -bins：分箱个数或边界；为整数表示等宽分箱；为序列表示自定义边界
#### -right：是否包含右端点，默认True；若为False则为左闭右开
#### -labels：为各区间提供标签，长度需与区间数一致

df2 = df.loc[:10, ['date','temp_max']] #Extract top 10 rows and 'date', 'temp_max' columns
df2['tm_level1'] = pd.cut(df2['temp_max'], bins=2) #Set 2 groups
print('DataFrame:\n', df2)
print('='*100)
print('Count each level:\n', df2['tm_level1'].value_counts())

DataFrame:
           date  temp_max     tm_level1
0   2012-01-01      12.8   (8.6, 12.8]
1   2012-01-02      10.6   (8.6, 12.8]
2   2012-01-03      11.7   (8.6, 12.8]
3   2012-01-04      12.2   (8.6, 12.8]
4   2012-01-05       8.9   (8.6, 12.8]
5   2012-01-06       4.4  (4.392, 8.6]
6   2012-01-07       7.2  (4.392, 8.6]
7   2012-01-08      10.0   (8.6, 12.8]
8   2012-01-09       9.4   (8.6, 12.8]
9   2012-01-10       6.1  (4.392, 8.6]
10  2012-01-11       6.1  (4.392, 8.6]
Count each level:
 tm_level1
(8.6, 12.8]     7
(4.392, 8.6]    4
Name: count, dtype: int64


In [ ]:
df2['tm_level2'] = pd.cut(df2['temp_max'], bins=[0, 7.5, 15]) #Set group cut points
print('DataFrame:\n', df2)
print('='*100)
print('Count each level:\n', df2['tm_level2'].value_counts())

DataFrame:
           date  temp_max     tm_level1    tm_level2
0   2012-01-01      12.8   (8.6, 12.8]  (7.5, 15.0]
1   2012-01-02      10.6   (8.6, 12.8]  (7.5, 15.0]
2   2012-01-03      11.7   (8.6, 12.8]  (7.5, 15.0]
3   2012-01-04      12.2   (8.6, 12.8]  (7.5, 15.0]
4   2012-01-05       8.9   (8.6, 12.8]  (7.5, 15.0]
5   2012-01-06       4.4  (4.392, 8.6]   (0.0, 7.5]
6   2012-01-07       7.2  (4.392, 8.6]   (0.0, 7.5]
7   2012-01-08      10.0   (8.6, 12.8]  (7.5, 15.0]
8   2012-01-09       9.4   (8.6, 12.8]  (7.5, 15.0]
9   2012-01-10       6.1  (4.392, 8.6]   (0.0, 7.5]
10  2012-01-11       6.1  (4.392, 8.6]   (0.0, 7.5]
Count each level:
 tm_level2
(7.5, 15.0]    7
(0.0, 7.5]     4
Name: count, dtype: int64


In [ ]:
df2 = pd.concat([df2, pd.cut(df2['temp_max'], bins=[0,7.5,15], labels=["L","H"])], axis=1) #Set bin labels
print('DataFrame:\n', df2)

DataFrame:
           date  temp_max     tm_level1    tm_level2 temp_max
0   2012-01-01      12.8   (8.6, 12.8]  (7.5, 15.0]        H
1   2012-01-02      10.6   (8.6, 12.8]  (7.5, 15.0]        H
2   2012-01-03      11.7   (8.6, 12.8]  (7.5, 15.0]        H
3   2012-01-04      12.2   (8.6, 12.8]  (7.5, 15.0]        H
4   2012-01-05       8.9   (8.6, 12.8]  (7.5, 15.0]        H
5   2012-01-06       4.4  (4.392, 8.6]   (0.0, 7.5]        L
6   2012-01-07       7.2  (4.392, 8.6]   (0.0, 7.5]        L
7   2012-01-08      10.0   (8.6, 12.8]  (7.5, 15.0]        H
8   2012-01-09       9.4   (8.6, 12.8]  (7.5, 15.0]        H
9   2012-01-10       6.1  (4.392, 8.6]   (0.0, 7.5]        L
10  2012-01-11       6.1  (4.392, 8.6]   (0.0, 7.5]        L


## Data Grouping and Aggregation

#### 基本语法：
##### df.groupby("分组字段")["要聚合的字段"].聚合函数()
##### df.groupby(["分组字段", "分组字段2", ...])[["要聚合的字段", "要聚合的字段2", ...]].聚合函数()
##### df.groupby(["分组字段", …])[["要聚合的字段", ...]].agg(['聚合函数', '聚合函数2', …])
##### df.groupby(["分组字段", …]).agg({"要聚合的字段":'聚合函数', "要聚合的字段":'聚合函数2'})

In [ ]:
## Exercise 6
import pandas as pd

#Read data
df = pd.read_csv("./Weather.csv")

In [ ]:
#Grouped by weather
#print(df.groupby('weather'))
print('Indices per group:\n', df.groupby('weather').groups)
print('='*100)
print('Sub-DataFrame ("fog" days):\n', df.groupby('weather').get_group('fog'))

Indices per group:
 {'drizzle': [0, 26, 45, 85, 103, 118, 135, 175, 186, 191, 193, 207, 208, 209, 213, 219, 221, 230, 231, 262, 263, 264, 269, 270, 282, 283, 284, 319, 329, 364, 365, 376, 381, 382, 383, 384, 385, 386, 387, 398, 406, 411, 432, 454, 472, 673], 'fog': [192, 260, 266, 267, 330, 433, 470, 481, 501, 550, 568, 574, 580, 593, 616, 622, 651, 654, 655, 680, 693, 734, 753, 800, 801, 916, 918, 921, 930, 948, 949, 977, 978, 993, 1000, 1001, 1008, 1009, 1010, 1011, 1012, 1035, 1042, 1066, 1077, 1078, 1089, 1090, 1093, 1098, 1101, 1102, 1103, 1107, 1108, 1109, 1111, 1115, 1116, 1120, 1121, 1123, 1125, 1126, 1137, 1139, 1142], 'rain': [1, 2, 3, 4, 5, 6, 8, 9, 20, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 50, 51, 52, 54, 55, 61, 63, 64, 68, 69, 70, 73, 75, 77, 78, 79, 80, 81, 84, 86, 87, 88, 89, 90, 91, 93, 96, 100, 101, 102, 105, 106, 107, 108, 109, 110, 112, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 129, 137, 138, 140, 141, 142, 143, 

In [ ]:
#Grouped by weather and calculate the precipitation std, mean, min and max by group
print('Precipitation standard deviation values per weather category:\n', df.groupby("weather")["precipitation"].std())
print('='*100)
print('Precipitation mean values per weather category:\n', df.groupby("weather")["precipitation"].mean())
print('='*100)
print('Precipitation minimum and maximum values per weather category:\n', df.groupby("weather")["precipitation"].agg(['min', 'max']))

Precipitation standard deviation values per weather category:
 weather
drizzle    0.000000
fog        0.000000
rain       7.990745
snow       7.021523
sun        0.000000
Name: precipitation, dtype: float64
Precipitation mean values per weather category:
 weather
drizzle    0.000000
fog        0.000000
rain       6.261248
snow       8.553846
sun        0.000000
Name: precipitation, dtype: float64
Precipitation minimum and maximum values per weather category:
          min   max
weather           
drizzle  0.0   0.0
fog      0.0   0.0
rain     0.0  54.1
snow     0.3  23.9
sun      0.0   0.0


In [ ]:
#Grouped by weather and calculate the temp_max mean and temp_min mean by group
print('Mean temp_max and temp_min values per weather category:\n', df.groupby('weather')[['temp_max', 'temp_min']].mean())

Mean temp_max and temp_min values per weather category:
           temp_max  temp_min
weather                     
drizzle  14.134783  6.056522
fog      16.538806  7.759701
rain     13.361437  7.520794
snow      5.573077  0.146154
sun      19.152245  8.859184


In [ ]:
#Grouped by weather and calculate the temp_max max and std, temp_min min and std by group
print(df.groupby("weather").agg({"temp_max":['max', 'std'], "temp_min":['min','std']}))

        temp_max           temp_min          
             max       std      min       std
weather                                      
drizzle     25.6  7.889634     -3.9  5.905051
fog         28.9  7.210135     -3.2  5.287093
rain        35.6  4.922874     -1.7  3.924981
snow        11.1  3.109155     -4.3  2.237182
sun         34.4  7.821213     -7.1  5.594023


#### 基本语法：
##### df.groupby(["分组字段", ...])[["要聚合的字段", ...]].apply(函数)
##### df.groupby(["分组字段", …])[["要聚合的字段", ...]].filter(条件)
##### df.groupby(["分组字段", …])[["要聚合的字段", ...]].transform(函数)

In [ ]:
def interval(ser):
    return ser.max()-ser.min()
#Grouped by weather and calculate the range between the highest and lowest temperatures by group
print('Maximum and Minimum temperature ranges per weather category:\n', df.groupby('weather')[['temp_max','temp_min']].apply(interval))

Maximum and Minimum temperature ranges per weather category:
          temp_max  temp_min
weather                    
drizzle      24.5      18.9
fog          23.9      21.0
rain         31.7      20.0
snow         12.2       9.9
sun          36.0      25.4


In [ ]:
#Filter weather groups with an average precipitation greater than 5
gs = df.groupby('weather').filter(lambda x: x['precipitation'].mean()>5)
print('Get weather with mean precipitation > 5:', gs['weather'].unique())
#print(df.groupby('weather')['precipitation'].mean()) #Test if the filtering is correct

Get weather with mean precipitation > 5: <StringArray>
['rain', 'snow']
Length: 2, dtype: str


In [ ]:
def normalize(ser):
    return (ser-ser.mean())/ser.std()
#Normalize precipitation within each weather group
df['pscale']=df.groupby('weather')['precipitation'].transform(normalize) #NaN may exist if std = 0
print(df)

            date  precipitation  temp_max  temp_min  wind  weather    pscale
0     2012-01-01            0.0      12.8       5.0   4.7  drizzle       NaN
1     2012-01-02           10.9      10.6       2.8   4.5     rain  0.580516
2     2012-01-03            0.8      11.7       7.2   2.3     rain -0.683447
3     2012-01-04           20.3      12.2       5.6   4.7     rain  1.756877
4     2012-01-05            1.3       8.9       2.8   6.1     rain -0.620874
...          ...            ...       ...       ...   ...      ...       ...
1456  2015-12-27            NaN       NaN       NaN   NaN      NaN       NaN
1457  2015-12-28            NaN       NaN       NaN   NaN      NaN       NaN
1458  2015-12-29            NaN       NaN       NaN   NaN      NaN       NaN
1459  2015-12-30            NaN       NaN       NaN   NaN      NaN       NaN
1460  2015-12-31           20.6      12.2       5.0   3.8     rain  1.794420

[1461 rows x 7 columns]


In [ ]:
print(df.groupby('weather')['precipitation'].agg(['mean', 'std'])) #View the mean and std for checking if the scale is correct

             mean       std
weather                    
drizzle  0.000000  0.000000
fog      0.000000  0.000000
rain     6.261248  7.990745
snow     8.553846  7.021523
sun      0.000000  0.000000
